# B0_SAM2_Unguided_Pipeline

Architecture:

1. Dataset Loader (Step 1 - 4)

        ↓

2. Unguided SAM2 automatic/grid generation (Step 5-8)

        ↓

3. Mask selection / filtering (identity operation in B0) (Step 9)

        ↓

4. Count (Step 10)

        ↓

5. Evaluation (Step 11)

        ↓

6. Results + diagnostics



# 1. Mount Google Drive, Respo and Setup

In [1]:
# 0 - mount google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1 - Define Datasets paths

from pathlib import Path

# ============================================================
# Project / Dataset Paths
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

DATA_ROOT = PROJECT_ROOT / "datasets"


# ------------------------------------------------------------
# FSC-147
# ------------------------------------------------------------

FSC147_ROOT = DATA_ROOT / "FSC147"
FSC147_RAW = FSC147_ROOT / "raw"

FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
FSC147_DENSITY_DIR = FSC147_RAW / "gt_density_map_adaptive_384_VarV2"

FSC147_ANNOTATION_FILE = FSC147_RAW / "annotation_FSC147_384.json"
FSC147_SPLIT_FILE = FSC147_RAW / "Train_Test_Val_FSC_147.json"
FSC147_CLASSES_FILE = FSC147_RAW / "ImageClasses_FSC147.txt"


# ------------------------------------------------------------
# OmniCount-191
# ------------------------------------------------------------

OMNICOUNT_ROOT = DATA_ROOT / "OmniCount-191"


# ------------------------------------------------------------
# Quick confirmation
# ------------------------------------------------------------

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)

print("\nFSC-147:")
print("FSC147_ROOT  :", FSC147_ROOT)
print("FSC147_RAW   :", FSC147_RAW)

print("\nOmniCount-191:")
print("OMNICOUNT_ROOT:", OMNICOUNT_ROOT)



PROJECT_ROOT : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
DATA_ROOT    : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets

FSC-147:
FSC147_ROOT  : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147
FSC147_RAW   : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147/raw

OmniCount-191:
OMNICOUNT_ROOT: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/OmniCount-191


In [3]:
# Clone Respo

%cd /content
!rm -rf Surrey-MScProject-Zero-Shot-Object-Counting
!git clone https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git



/content
Cloning into 'Surrey-MScProject-Zero-Shot-Object-Counting'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 21 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 11.15 KiB | 5.58 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [4]:
# Import Setup

import sys

repo_path = "/content/Surrey-MScProject-Zero-Shot-Object-Counting"

if repo_path not in sys.path:
    sys.path.append(repo_path)

from src.datasets import (
    FSC147Dataset,
    OmniCount191Dataset,
    collate_keep_dicts,
)

from src.evaluation import evaluate_single_label_counts

from torch.utils.data import DataLoader



# 2. Unguided SAM2 automatic/grid generation

In [5]:
# ============================================================
# Step 5 - Initialise Unguided SAM2
# ============================================================

import torch
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(
    "Device:",
    torch.cuda.get_device_name(0)
    if device.type == "cuda"
    else "CPU"
)

# ------------------------------------------------------------
# Load SAM 2.1 from Meta github
# ------------------------------------------------------------

print("Loading SAM 2.1...")

%cd /content

if not Path("/content/sam2").exists():
    !git clone https://github.com/facebookresearch/sam2.git

%cd /content/sam2

!pip install -q -e ".[notebooks]"

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

print("SAM 2.1 loaded.")

# ------------------------------------------------------------
# SAM2.1 Hiera-Small configuration
# ------------------------------------------------------------

SAM2_ROOT = Path("/content/sam2")

MODEL_CFG = "configs/sam2.1/sam2.1_hiera_s.yaml"
CHECKPOINT = SAM2_ROOT / "checkpoints" / "sam2.1_hiera_small.pt"

print("Model config :", MODEL_CFG)
print("Checkpoint   :", CHECKPOINT)

# Download SAM2.1 Hiera-Small checkpoint if needed

if not CHECKPOINT.exists():
    print("Downloading SAM2.1 Hiera-Small checkpoint...")

    !mkdir -p /content/sam2/checkpoints
    !wget -q \
        https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt \
        -O /content/sam2/checkpoints/sam2.1_hiera_small.pt

print("Checkpoint exists:", CHECKPOINT.exists())

# ------------------------------------------------------------
# Build SAM2.1 Hiera-Small
# ------------------------------------------------------------
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

sam2_model = build_sam2(
    MODEL_CFG,
    str(CHECKPOINT),
    device=device
)

predictor = SAM2ImagePredictor(sam2_model)

mask_generator = SAM2AutomaticMaskGenerator(sam2_model)       #B0 - Use SAM2AutomaticMaskGenerator


print("\nSAM2.1 Hiera-Small loaded successfully")
print("Device:", device)
print("Predictor:", type(predictor).__name__)
print("Mask generator:", type(mask_generator).__name__)


Device: Tesla T4
Loading SAM 2.1...
/content
Cloning into 'sam2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 1107 (delta 10), reused 4 (delta 4), pack-reused 1093 (from 2)
Receiving objects: 100% (1107/1107), 134.85 MiB | 37.65 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# ============================================================
# Step 6 - B0 Unguided SAM2 Mask Generation
# Diagnostic mode / Full evaluation mode
# ============================================================

import numpy as np
import time

start_time = time.time()

# ------------------------------------------------------------
# Experiment mode
# ------------------------------------------------------------

MODE = "evaluation"     # "diagnostic" or "evaluation"

if MODE not in {"diagnostic", "evaluation"}:
    raise ValueError("MODE must be 'diagnostic' or 'evaluation'")

# Diagnostic mode:
#   - small subset
#   - retain images, masks and full SAM2 annotations
#   - used for Steps 7-8
#
# Evaluation mode:
#   - full split
#   - retain only lightweight information needed for counting/evaluation
#   - avoids storing full-resolution masks for all images

MAX_IMAGES = 10 if MODE == "diagnostic" else None


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATASET_NAME = "fsc147"   # "fsc147" or "omnicount191"
SPLIT = "val"

if DATASET_NAME == "fsc147":

    b0_dataset = FSC147Dataset(
        root=FSC147_RAW,
        split=SPLIT,
        load_images=True
    )

elif DATASET_NAME == "omnicount191":

    b0_dataset = OmniCount191Dataset(
        root=OMNICOUNT_ROOT,
        split=SPLIT,
        load_images=True
    )

else:
    raise ValueError(f"Unsupported dataset: {DATASET_NAME}")


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

b0_loader = DataLoader(
    b0_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_keep_dicts
)


# ------------------------------------------------------------
# Progress information
# ------------------------------------------------------------

progress_total = (
    min(MAX_IMAGES, len(b0_dataset))
    if MAX_IMAGES is not None
    else len(b0_dataset)
)

print(
    f"Mode: {MODE} | "
    f"Dataset: {DATASET_NAME} | "
    f"Split: {SPLIT} | "
    f"Images to process: {progress_total}"
)


# ------------------------------------------------------------
# Run unguided native Meta SAM2
# ------------------------------------------------------------

raw_results = []

for image_idx, batch in enumerate(b0_loader):

    if MAX_IMAGES is not None and image_idx >= MAX_IMAGES:
        break

    sample = batch[0]
    image = sample["image"]

    print(
        f"[{image_idx + 1}/{progress_total}] "
        f"{sample['image_id']}"
    )

    # Convert PIL image to HWC uint8 NumPy array
    image_np = np.array(
        image.convert("RGB"),
        dtype=np.uint8,
        copy=True
    )

    # Native SAM2 automatic mask generation
    outputs = mask_generator.generate(image_np)

    num_raw_masks = len(outputs)

    # ========================================================
    # Diagnostic mode
    # ========================================================

    if MODE == "diagnostic":

        masks = [
            ann["segmentation"]
            for ann in outputs
        ]

        predicted_ious = [
            ann["predicted_iou"]
            for ann in outputs
        ]

        stability_scores = [
            ann["stability_score"]
            for ann in outputs
        ]

        result = {
            "image_id": sample["image_id"],
            "image_path": sample["image_path"],
            "dataset": sample["dataset"],
            "split": sample["split"],
            "domain": sample["domain"],
            "width": sample["width"],
            "height": sample["height"],
            "categories": sample["categories"],

            # Development / diagnostic only
            "image": image,
            "raw_annotations": outputs,
            "raw_masks": masks,
            "raw_predicted_ious": predicted_ious,
            "raw_stability_scores": stability_scores,

            "num_raw_masks": num_raw_masks,
        }

    # ========================================================
    # Evaluation mode
    # ========================================================

    else:

        result = {
            "image_id": sample["image_id"],
            "image_path": sample["image_path"],
            "dataset": sample["dataset"],
            "split": sample["split"],
            "domain": sample["domain"],
            "width": sample["width"],
            "height": sample["height"],
            "categories": sample["categories"],

            # Only retain the information required downstream
            "num_raw_masks": num_raw_masks,
        }

    raw_results.append(result)

    print(
        f"    Generated masks: {num_raw_masks}"
    )


elapsed_time = time.time() - start_time

print(
    f"\nFinished SAM2 inference on "
    f"{len(raw_results)} images."
)

print(f"Total inference time: {elapsed_time / 60:.2f} minutes")
print(
    f"Average time per image: "
    f"{elapsed_time / len(raw_results):.2f} seconds"
)

Mode: evaluation | Dataset: fsc147 | Split: val | Images to process: 1286
[1/1286] 190.jpg
    Generated masks: 2
[2/1286] 191.jpg
    Generated masks: 2
[3/1286] 192.jpg
    Generated masks: 3
[4/1286] 194.jpg
    Generated masks: 44
[5/1286] 195.jpg
    Generated masks: 18
[6/1286] 196.jpg
    Generated masks: 66
[7/1286] 197.jpg
    Generated masks: 66
[8/1286] 198.jpg
    Generated masks: 81
[9/1286] 214.jpg
    Generated masks: 30
[10/1286] 215.jpg
    Generated masks: 103
[11/1286] 216.jpg
    Generated masks: 12
[12/1286] 217.jpg
    Generated masks: 49
[13/1286] 218.jpg
    Generated masks: 29
[14/1286] 219.jpg
    Generated masks: 26
[15/1286] 220.jpg
    Generated masks: 20
[16/1286] 221.jpg
    Generated masks: 25
[17/1286] 222.jpg
    Generated masks: 50
[18/1286] 224.jpg
    Generated masks: 13
[19/1286] 226.jpg
    Generated masks: 43
[20/1286] 227.jpg
    Generated masks: 21
[21/1286] 228.jpg
    Generated masks: 5
[22/1286] 229.jpg
    Generated masks: 22
[23/1286] 231.

# Step 7 — Raw SAM2 Output Sanity Checks

In [7]:
# Visualize SAM2 masks function - Call only if needed

import matplotlib.pyplot as plt


def visualize_sam2_masks(
    image,
    masks,
    title=None
):
    """
    Visualize SAM2 masks for diagnostic purposes only.
    """

    image_np = np.array(image)

    plt.figure(figsize=(10, 10))
    plt.imshow(image_np)

    ax = plt.gca()
    ax.set_autoscale_on(False)

    for mask in masks:

        m = np.asarray(mask) > 0

        overlay = np.zeros(
            (m.shape[0], m.shape[1], 4),
            dtype=float
        )

        overlay[m, :3] = np.random.random(3)
        overlay[m, 3] = 0.5

        ax.imshow(overlay)

    plt.axis("off")

    if title is None:
        title = (
            f"SAM2 unguided: "
            f"{len(masks)} masks"
        )

    plt.title(title)
    plt.show()

In [8]:
# Inspect results - sample [0, 3, 9]

if MODE != "diagnostic":
    print("Step 7 skipped: diagnostic mode only.")
else:
    for idx in [0, 3, 9]:

        r = raw_results[idx]

        print(r.keys())

        print("Masks:", len(r["raw_masks"]))
        print(
            "Predicted IoUs:",
            len(r["raw_predicted_ious"])
        )
        print(
            "Stability scores:",
            len(r["raw_stability_scores"])
        )

        print(
            "First predicted IoUs:",
            r["raw_predicted_ious"][:5]
        )
        print(
            "First stability scores:",
            r["raw_stability_scores"][:5]
        )
        print("\n====================================================================")


Step 7 skipped: diagnostic mode only.


In [9]:
# Inspect native Meta SAM2 output structure

if MODE != "diagnostic":
    print("Step 7 skipped: diagnostic mode only.")
else:
    # existing Step 7 code
    print("Output type:", type(outputs))
    print("Number of annotations:", len(outputs))

    if len(outputs) > 0:
        print("\nKeys in first annotation:")
        print(outputs[0].keys())

        print("\nFirst annotation (excluding segmentation):")
        for key, value in outputs[0].items():
            if key != "segmentation":
                print(f"{key}: {value}")

Step 7 skipped: diagnostic mode only.


In [10]:
# Inspect GT points - ONLY valid for FSC-147

if MODE != "diagnostic":
    print("Step 7 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "GT-point inspection skipped: "
        "FSC-147 only."
    )

else:
    # existing Step 7 code
    for idx in [0,3,9]:

      r = raw_results[idx]

      print("\n=================================")
      print("Image:", r["image_id"])
      print("Categories:", r["categories"])
      print("Raw masks:", r["num_raw_masks"])
      print("---------------------------------")

      # FSC-147 GT points

      category_name = next(iter(r["categories"]))
      gt_points = r["categories"][category_name]["points"]

      gt_count = len(gt_points)

      print("Target class:", category_name)
      print("GT count:", gt_count)
      print("SAM2 raw masks:", r["num_raw_masks"])


Step 7 skipped: diagnostic mode only.


In [11]:
# Visual sanity check

if MODE != "diagnostic":
    print("Step 7 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "GT-point inspection skipped: "
        "FSC-147 only."
    )

else:
    # existing Step 7 code
    for idx in [0, 3, 9]:

        r = raw_results[idx]

        visualize_sam2_masks(
            r["image"],
            r["raw_masks"],
            title=(
                f"{r['image_id']} | "
                f"Raw SAM2 masks: {r['num_raw_masks']}"
            )
        )

Step 7 skipped: diagnostic mode only.


# Step 8 — Raw SAM2 Mask Diagnostics

Note: The whole Step 8 is for diagnosis and 8.3 - 8.7 run on FSC-147 ONLY

Step 8.1 - Calculate mask area and area ratio

In [12]:
# sanity check for single image and single mask

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")
else:
    # existing Step 8 code
    image_idx = 0
    mask_idx = 0

    r = raw_results[image_idx]

    ann = r['raw_annotations'][mask_idx]

    mask_area = ann['area']
    image_width = r['width']
    image_height = r['height']
    total_image_area = image_width * image_height
    mask_area_ratio = mask_area / total_image_area

    print(f"Image: {r['image_id']}")
    print(f"Image size: {image_width} x {image_height}")
    print(f"Mask index: {mask_idx}")
    print(f"Mask area: {mask_area}")
    print(f"Mask area ratio: {mask_area_ratio}")

Step 8 skipped: diagnostic mode only.


In [13]:
# ============================================================
# Step 8.1 - Calculate mask area and area ratio for all masks
# ============================================================

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")
else:
    # existing Step 8 code
    mask_diagnostics = []

    for image_idx, r in enumerate(raw_results):

        image_width = r["width"]
        image_height = r["height"]
        total_image_area = image_width * image_height

        for mask_idx, ann in enumerate(r["raw_annotations"]):

            mask_area = ann["area"]
            mask_area_ratio = mask_area / total_image_area

            mask_diagnostics.append({
                "image_idx": image_idx,
                "image_id": r["image_id"],
                "mask_idx": mask_idx,
                "mask_area": mask_area,
                "mask_area_ratio": mask_area_ratio,
            })

    print("Images processed:", len(raw_results))
    print("Total masks:", len(mask_diagnostics))

Step 8 skipped: diagnostic mode only.


In [14]:
# ============================================================
# Step 8.2 - Add SAM2 quality diagnostics
# ============================================================

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")
else:
    # existing Step 8 code
    for d in mask_diagnostics:

        r = raw_results[d["image_idx"]]
        ann = r["raw_annotations"][d["mask_idx"]]

        d["predicted_iou"] = ann["predicted_iou"]
        d["stability_score"] = ann["stability_score"]
        d["point_coords"] = ann["point_coords"]

    print("Diagnostics updated:", len(mask_diagnostics))

Step 8 skipped: diagnostic mode only.


In [15]:
if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")
else:
    for d in mask_diagnostics[:5]:
        print(d)

Step 8 skipped: diagnostic mode only.


In [16]:
# ============================================================
# Step 8.3 - Count FSC-147 GT target points inside each mask
# Note: Step 8.3 - 8.7 are vaild ONLY for FSC-147
# ============================================================

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing Step 8 code
    for d in mask_diagnostics:

        r = raw_results[d["image_idx"]]
        ann = r["raw_annotations"][d["mask_idx"]]

        mask = np.asarray(ann["segmentation"], dtype=bool)

        # FSC-147 has one target category per image
        category_name = next(iter(r["categories"]))
        gt_points = r["categories"][category_name]["points"]

        points_inside = 0

        for x, y in gt_points:

            # Convert annotation coordinates to image pixel indices
            x_idx = int(round(x))
            y_idx = int(round(y))

            # Clip only for safe array indexing
            x_idx = np.clip(x_idx, 0, mask.shape[1] - 1)
            y_idx = np.clip(y_idx, 0, mask.shape[0] - 1)

            if mask[y_idx, x_idx]:
                points_inside += 1

        d["num_gt_points_inside"] = points_inside

    print("GT-point diagnostics updated:", len(mask_diagnostics))

Step 8 skipped: diagnostic mode only.


In [17]:
if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing Step 8 code
    for d in mask_diagnostics[:10]:
        print(
            d["image_id"],
            "mask", d["mask_idx"],
            "GT points inside:", d["num_gt_points_inside"]
        )

Step 8 skipped: diagnostic mode only.


In [18]:
if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing Step 8 code
    n_zero = sum(d["num_gt_points_inside"] == 0 for d in mask_diagnostics)
    n_one = sum(d["num_gt_points_inside"] == 1 for d in mask_diagnostics)
    n_multiple = sum(d["num_gt_points_inside"] > 1 for d in mask_diagnostics)

    print("Masks with 0 GT points:", n_zero)
    print("Masks with 1 GT point :", n_one)
    print("Masks with >1 GT point:", n_multiple)

Step 8 skipped: diagnostic mode only.


In [19]:
# ============================================================
# Step 8.4 - Calculate GT-point coverage for each image
# ============================================================

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing Step 8 code
    image_coverage = []

    for image_idx, r in enumerate(raw_results):

        category_name = next(iter(r["categories"]))
        gt_points = r["categories"][category_name]["points"]

        num_gt = len(gt_points)

        covered = np.zeros(num_gt, dtype=bool)

        for ann in r["raw_annotations"]:

            mask = np.asarray(ann["segmentation"], dtype=bool)

            for gt_idx, (x, y) in enumerate(gt_points):

                x_idx = int(round(x))
                y_idx = int(round(y))

                x_idx = np.clip(x_idx, 0, mask.shape[1] - 1)
                y_idx = np.clip(y_idx, 0, mask.shape[0] - 1)

                if mask[y_idx, x_idx]:
                    covered[gt_idx] = True

        num_covered = covered.sum()

        coverage_ratio = (
            num_covered / num_gt
            if num_gt > 0
            else 0.0
        )

        image_coverage.append({
            "image_idx": image_idx,
            "image_id": r["image_id"],
            "category": category_name,
            "gt_count": num_gt,
            "gt_points_covered": int(num_covered),
            "gt_points_missed": int(num_gt - num_covered),
            "coverage_ratio": coverage_ratio,
            "num_raw_masks": r["num_raw_masks"],
        })

Step 8 skipped: diagnostic mode only.


In [20]:
if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing code
    for d in image_coverage:
        print(
            d["image_id"],
            "| GT:", d["gt_count"],
            "| Covered:", d["gt_points_covered"],
            "| Missed:", d["gt_points_missed"],
            "| Coverage:", f'{d["coverage_ratio"]:.3f}',
            "| Raw masks:", d["num_raw_masks"],
        )

Step 8 skipped: diagnostic mode only.


In [21]:
# ============================================================
# Step 8.5 - Overall raw-mask diagnostic summary
# ============================================================

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing Step 8 code
    total_gt = sum(d["gt_count"] for d in image_coverage)
    total_covered = sum(d["gt_points_covered"] for d in image_coverage)
    total_missed = sum(d["gt_points_missed"] for d in image_coverage)

    overall_coverage = total_covered / total_gt

    print("Total GT points:", total_gt)
    print("GT points covered:", total_covered)
    print("GT points missed:", total_missed)
    print(f"Overall GT-point coverage: {overall_coverage:.3f}")

    print("\nRaw mask diagnostics:")
    print("Total raw masks:", len(mask_diagnostics))
    print("Masks with 0 GT points:", n_zero)
    print("Masks with 1 GT point:", n_one)
    print("Masks with >1 GT point:", n_multiple)

Step 8 skipped: diagnostic mode only.


In [22]:
# ============================================================
# Step 8.6 - Compare mask properties by GT-point content
# ============================================================

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing code
    import pandas as pd

    diagnostics_df = pd.DataFrame(mask_diagnostics)

    diagnostics_df["gt_group"] = diagnostics_df["num_gt_points_inside"].apply(
        lambda n: "0 GT"
        if n == 0
        else "1 GT"
        if n == 1
        else ">1 GT"
    )

    summary = diagnostics_df.groupby("gt_group")[
        [
            "mask_area_ratio",
            "predicted_iou",
            "stability_score"
        ]
    ].agg(["count", "mean", "median", "min", "max"])

    display(summary)

Step 8 skipped: diagnostic mode only.


In [23]:
# ============================================================
# Step 8.7 - Visualise diagnostic distributions
# ============================================================

if MODE != "diagnostic":
    print("Step 8 skipped: diagnostic mode only.")

elif DATASET_NAME != "fsc147":
    print(
        "Step 8.3-8.7 skipped: "
        "FSC-147 GT-point diagnostics only."
    )

else:
    # existing code
    import matplotlib.pyplot as plt

    diagnostic_features = [
        "mask_area_ratio",
        "predicted_iou",
        "stability_score",
    ]

    for feature in diagnostic_features:

        ax = diagnostics_df.boxplot(
            column=feature,
            by="gt_group",
            grid=False,
            figsize=(8, 5)
        )

        plt.title(feature)
        plt.suptitle("")
        plt.xlabel("GT-point content")
        plt.ylabel(feature)
        plt.show()

Step 8 skipped: diagnostic mode only.


# Step 9 - Mask selection / filtering

In [24]:
# ============================================================
# Step 9 - Mask selection / filtering
# B0: no additional filtering
# ============================================================

for r in raw_results:

    if MODE == "diagnostic":

        r["selected_annotations"] = r["raw_annotations"]
        r["selected_masks"] = r["raw_masks"]
        r["num_selected_masks"] = len(r["selected_masks"])

    else:

        # B0 performs no additional filtering,
        # therefore selected count = native SAM2 returned count
        r["num_selected_masks"] = r["num_raw_masks"]


print("Step 9 complete: no additional B0 filtering applied.")

Step 9 complete: no additional B0 filtering applied.


In [25]:
# ============================================================
# Step 10 - Convert selected masks to predicted count
# ============================================================


for r in raw_results:
    # B0 produces one class-agnostic count per image.
    r["pred_count"] = r["num_selected_masks"]

print("Step 10 complete: predicted counts generated.")

if DATASET_NAME == "fsc147":
    for r in raw_results:
        category_name = next(iter(r["categories"]))
        gt_count = r["categories"][category_name]["count"]

        print(
            f'{r["image_id"]} | '
            f'Target: {category_name} | '
            f'GT: {gt_count} | '
            f'Predicted: {r["pred_count"]}'
        )
else:
    for r in raw_results:
        print(
            f'{r["image_id"]} | '
            f'Class-agnostic predicted count: {r["pred_count"]}'
        )

    print("\nMulti-label GT evaluation: not applicable for B0.")



Step 10 complete: predicted counts generated.
190.jpg | Target: seagulls | GT: 13 | Predicted: 2
191.jpg | Target: seagulls | GT: 15 | Predicted: 2
192.jpg | Target: seagulls | GT: 19 | Predicted: 3
194.jpg | Target: peaches | GT: 82 | Predicted: 44
195.jpg | Target: peaches | GT: 10 | Predicted: 18
196.jpg | Target: peaches | GT: 85 | Predicted: 66
197.jpg | Target: peaches | GT: 77 | Predicted: 66
198.jpg | Target: peaches | GT: 69 | Predicted: 81
214.jpg | Target: grapes | GT: 64 | Predicted: 30
215.jpg | Target: grapes | GT: 259 | Predicted: 103
216.jpg | Target: grapes | GT: 46 | Predicted: 12
217.jpg | Target: grapes | GT: 60 | Predicted: 49
218.jpg | Target: grapes | GT: 58 | Predicted: 29
219.jpg | Target: grapes | GT: 47 | Predicted: 26
220.jpg | Target: grapes | GT: 59 | Predicted: 20
221.jpg | Target: grapes | GT: 42 | Predicted: 25
222.jpg | Target: grapes | GT: 65 | Predicted: 50
224.jpg | Target: grapes | GT: 20 | Predicted: 13
226.jpg | Target: grapes | GT: 78 | Predicte

In [26]:
if DATASET_NAME != "fsc147":
    print(
        "Step 11 skipped: "
        "current evaluator is for FSC-147 single-label counting."
    )

else:
    evaluation_df, metrics = evaluate_single_label_counts(
        raw_results
    )

    print("============================================================")
    print("B0 Quantitative Evaluation - FSC-147 Validation Set")
    print("============================================================")

    print(f'Number of images : {metrics["num_samples"]}')
    print(f'MAE              : {metrics["mae"]:.4f}')
    print(f'RMSE             : {metrics["rmse"]:.4f}')

    print("\nError direction summary")
    print("-----------------------")
    print(f'Undercount images : {metrics["num_undercount"]}')
    print(f'Exact images      : {metrics["num_exact"]}')
    print(f'Overcount images  : {metrics["num_overcount"]}')
    print(
        f'Mean signed error : '
        f'{metrics["mean_signed_error"]:.4f}'
    )

    display(evaluation_df)



B0 Quantitative Evaluation - FSC-147 Validation Set
Number of images : 1286
MAE              : 41.6400
RMSE             : 123.7591

Error direction summary
-----------------------
Undercount images : 921
Exact images      : 36
Overcount images  : 329
Mean signed error : -38.1345


,image_id,category,gt_count,pred_count,error,abs_error,squared_error
0,190.jpg,seagulls,13,2,-11,11,121
1,191.jpg,seagulls,15,2,-13,13,169
2,192.jpg,seagulls,19,3,-16,16,256
3,194.jpg,peaches,82,44,-38,38,1444
4,195.jpg,peaches,10,18,8,8,64
...,...,...,...,...,...,...,...
1281,6916.jpg,books,65,50,-15,15,225
1282,7122.jpg,books,21,7,-14,14,196
1283,6798.jpg,birds,87,12,-75,75,5625
1284,7011.jpg,birds,44,16,-28,28,784


In [27]:
if DATASET_NAME == "fsc147":

    OUTPUT_ROOT = PROJECT_ROOT / "Outputs"
    B0_OUTPUT_DIR = OUTPUT_ROOT / "B0"

    B0_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    B0_CSV = (
        B0_OUTPUT_DIR
        / "B0_FSC147_val_evaluation.csv"
    )

    evaluation_df.to_csv(
        B0_CSV,
        index=False
    )

    print("Saved:", B0_CSV)

Saved: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/Outputs/B0/B0_FSC147_val_evaluation.csv
